# Trabalho Prático - Notebook do grupo

**Inteligência Artificial - Graduação** · Prof. Bruno Nogueira · FACOM/UFMS · 2026/2

**Grupo:** Isabela H. Nakano, João G. de S. Balbuena, Lara P. Santos, Nayla P. Figueiredo · **Tema:** Rui — Chatbot sobre o Restaurante Universitário (RU) da UFMS

---

Este é o notebook **entregável** do grupo. Ele precisa rodar do início ao fim
("Kernel → Restart & Run All") e **regerar os modelos** salvos em `modelos/`. A
estrutura segue o `00_exemplo_biblioteca.ipynb`; consulte-o sempre que tiver
dúvida.

Marcadores usados aqui:

* `🔨 DONE` - algo que o grupo precisa fazer/decidir;
* `📝 RELATÓRIO` - um resultado que deve aparecer (e ser discutido) no artigo.

Requisitos mínimos de cada seção estão no enunciado. Pode acrescentar seções,
células e análises à vontade - só não remova as seções numeradas.

---
## 0. Ambiente

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# 🔨 DONE (só para o Colab): endereço do repositório do grupo
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"

if "google.colab" in sys.modules:
    pasta = Path("/content") / Path(REPO_URL).stem
    if not pasta.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(pasta)], check=True)
    os.chdir(pasta)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

raiz = Path.cwd()
while not (raiz / "chatbot").is_dir():
    if raiz.parent == raiz:
        raise RuntimeError("Rode este notebook de dentro do repositório do trabalho.")
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz))

In [2]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from chatbot.avaliacao import avaliar, tabela_resumo
from chatbot.config import carregar_config
from chatbot.dados import adicionar_escopo, carregar_dados, resumo, validar_dados, vazamento
from chatbot.grafo import Conversa, construir_grafo
from chatbot.modelos import carregar_classificadores, salvar_classificadores
from chatbot.representacoes import tfidf_caracteres, tfidf_palavras, EmbeddingsSentenca
from chatbot.rotulos import CLASSES_ESCOPO

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CORES = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
pd.set_option("display.max_colwidth", 80)

---
## 1. Configuração

🔨 DONE: preencha o `grupo.yaml` da raiz do repositório (tema e intenções
**validados com o professor**) antes de rodar esta seção.

In [3]:
config = carregar_config("grupo.yaml")
print("Grupo:", config.grupo, "| Tema:", config.tema)
pd.DataFrame([(i.nome, i.descricao) for i in config.intencoes.values()], columns=["intenção", "descrição"])

Grupo: Corporação | Tema: Rui - O seu assistente virtual do RU da UFMS


,intenção,descrição
0,cardapio,Perguntas a respeito do cardápio do dia/semana para almoço/jantar
1,horarios,Perguntas a respeito dos horários de funcionamento do RU
2,valores,"Perguntas a respeito dos preços das refeições no RU para cada categoria, inc..."
3,fichas,Perguntas a respeito de como funciona o processo de adquirir e usar fichas p...
4,subsidio,Perguntas a respeito de como funciona o acesso ao subsídio para comer no RU....
5,geral,"Perguntas gerais sobre o RU, como: contatos, recursos disponíveis, localizaç..."


---
## 2. Dados

## 2. Dados

Parte dos dados de treino (276 frases) foi gerada por LLM (`openai/gpt-6-luna` via OpenRouter e ChatGPT versão gratuita), usando prompts em estilos variados: formal, informal (como no WhatsApp), com erros de digitação, sem pontuação, apressado e muito curto, todos registrados em `dados/prompts_llm.md`[cite: 12, 13]. Todas essas frases foram revisadas manualmente para remover repetições e corrigir rótulos[cite: 12]. O restante foi escrito pelos membros do grupo (701 frases), mantendo o equilíbrio entre os integrantes para evitar vieses, além de 100 frases fora do domínio retiradas da base `dados/geral/massive_fora_dominio.csv` (também revisadas)[cite: 12, 13].

Os negativos difíceis foram pensados usando contextos e palavras-chave que poderiam induzir o classificador ao erro, com foco em outros assuntos e setores da UFMS[cite: 12, 13]. Exemplos: "que horas abre a facom?", "O subsídio pra receber auxílio permanência funciona como?" e "Qual o contato da AGETIC?"[cite: 13].

Os dados de teste foram coletados a partir de um formulário respondido por colegas e familiares externos ao grupo, excluindo frases repetidas que já estavam no treino e garantindo que nenhum membro inserisse frases próprias no teste (100% origem humana e sem vazamento)[cite: 13].

A tabela abaixo mostra a distribuição dos dados de treino por intenção e origem:


| **Nome**           | **Cardápio** | **Valores** | **Horários** | **Fichas** | **Subsídio** | **Geral** | **Fora do Domínio** | **Chitchat** | **Total ** |
|:------------------:|:------------:|:-----------:|:------------:|:----------:|:------------:|:---------:|:-------------------:|:------------:|:----------:|
| **Isabela Nakano** | 14           | 11          | 13           | 13         | 15           | 17        | 50                  | 42           | 175        |
| **João Balbuena**  | 11           | 12          | 12           | 10         | 14           | 18        | 51                  | 45           | 173        |
| **Lara Pereira**   | 10           | 12          | 13           | 11         | 14           | 13        | 53                  | 50           | 176        |
| **Nayla Peralta**  | 14           | 10          | 12           | 12         | 13           | 14        | 49                  | 53           | 177        |
| **Total Humano**   | 49           | 45          | 50           | 46         | 56           | 62        | 203                 | 190          | 701        |
| **Massive**        | --           | --          | --           | --         | --           | --        | 100                 | 0            | 100        |
| **LLM**            | 49           | 48          | 43           | 49         | 41           | 26        | 20                  | 0            | 276        |
| **Total**          | 98           | 93          | 93           | 95         | 97           | 88        | 323                 | 190          | 1077       |




(Tabela gerada com auxílio do site [Table Convert](https://tableconvert.com/pt/markdown-generator))


📝 RELATÓRIO: tabela intenção × origem do treino e do teste.

In [4]:
treino = adicionar_escopo(carregar_dados(config.treino), config)
teste = adicionar_escopo(carregar_dados(config.teste), config)
print("Problemas no treino:", validar_dados(treino, config) or "nenhum")
print("Problemas no teste: ", validar_dados(teste, config) or "nenhum")
print("Vazamento treino→teste:", vazamento(treino, teste) or "nenhum")
display(resumo(treino))
display(resumo(teste))

Problemas no treino: nenhum
Problemas no teste:  nenhum
Vazamento treino→teste: nenhum


origem,humano,llm,massive,total
intencao,,,,
cardapio,49,49,0,98
chitchat,190,0,0,190
fichas,46,49,0,95
fora_dominio,203,20,100,323
geral,62,26,0,88
horarios,50,43,0,93
subsidio,56,41,0,97
valores,45,48,0,93
total,701,276,100,1077


origem,humano,total
intencao,,
cardapio,47,47
chitchat,63,63
fichas,39,39
fora_dominio,40,40
geral,27,27
horarios,38,38
subsidio,40,40
valores,35,35
total,329,329


### Gerando frases de treino com o LLM (opcional)

Rode no terminal (ou descomente a célula abaixo). As frases vão para
`dados/gerados_llm.csv` e os prompts ficam registrados em `dados/prompts_llm.md`.
**Revise tudo** antes de copiar para o `treino.csv`; nunca use no teste; declare no
`USO_DE_IA.md`.

In [ ]:
# Para rodar daqui, tire o "#" das linhas abaixo (gasta créditos da chave do grupo):
# subprocess.run([sys.executable, "scripts/gerar_dados_llm.py", "--n", "10"], check=True)
# gerados = pd.read_csv("dados/gerados_llm.csv")
# gerados.sample(15)

🔨 DONE: análise exploratória - tamanho das mensagens por classe, palavras mais
frequentes por intenção, exemplos de negativos difíceis, diferenças entre frases
humanas e de LLM...

FEITA com o script analise_exploratoria.py, que gera a pasta figuras_eda

---
## 3. Representações de texto

🔨 DONE: escolha **pelo menos duas** representações para comparar. Os construtores
de `chatbot/representacoes.py` aceitam os parâmetros do `TfidfVectorizer`
(`ngram_range`, `min_df`, `stop_words`, ...). Embeddings pré-treinados
(`EmbeddingsSentenca`) são opcionais.

In [5]:
REPRESENTACOES = {
    "TF-IDF palavras": tfidf_palavras,
    "TF-IDF caracteres": tfidf_caracteres,
    # "Embeddings": EmbeddingsSentenca, #usado para comparação -> performance pior que o TF-IDF
    # 🔨 DONE: outras variações (ex.: com stopwords, só unigramas, embeddings...)
}

---
## 4. Classificador de escopo (dominio / chitchat / fora_dominio)

🔨 DONE: compare **pelo menos três algoritmos vistos em aula**, com validação
cruzada estratificada **só no treino**. Classificadores que ainda serão
apresentados na disciplina (KNN, Naive Bayes, SVM...) podem entrar quando forem
vistos.

📝 RELATÓRIO: tabela/gráfico da comparação, com média e desvio entre folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

CLASSIFICADORES = {
    "Regressão logística": lambda: LogisticRegression(C=10, max_iter=2000),
    "Rede neural (MLP)": lambda: MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=600,
                                               random_state=RANDOM_STATE),
    "Árvore de decisão": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
    # 🔨 DONE: acrescente outros
}


def comparar(textos, rotulos):
    linhas = []
    for nome_rep, rep in REPRESENTACOES.items():
        for nome_clf, clf in CLASSIFICADORES.items():
            pipe = Pipeline([("vetorizador", rep()), ("classificador", clf())])
            r = cross_validate(pipe, textos, rotulos, cv=cv, scoring=["f1_macro", "accuracy"],
                               return_train_score=True, n_jobs=1)
            linhas.append({"representação": nome_rep, "classificador": nome_clf,
                           "F1 macro (CV)": r["test_f1_macro"].mean(), "desvio": r["test_f1_macro"].std(),
                           "acurácia (CV)": r["test_accuracy"].mean(),
                           "F1 macro (treino)": r["train_f1_macro"].mean()})
    return pd.DataFrame(linhas).sort_values("F1 macro (CV)", ascending=False).reset_index(drop=True)


tab_escopo = comparar(treino["texto"], treino["escopo"])
tab_escopo.round(3)

In [ ]:
def grafico_comparacao(tab, titulo):
    piv = tab.pivot(index="classificador", columns="representação", values="F1 macro (CV)")
    err = tab.pivot(index="classificador", columns="representação", values="desvio")
    ax = piv.plot.barh(xerr=err, color=CORES[:2], figsize=(7, 3.4), capsize=3)
    ax.set_xlabel("F1 macro (validação cruzada, 5 folds)")
    ax.set_xlim(0, 1)
    ax.set_title(titulo)
    ax.legend(loc="lower left")
    plt.tight_layout()
    plt.show()


grafico_comparacao(tab_escopo, "Classificador de escopo")

🔨 DONE: ajuste de hiperparâmetros dos melhores candidatos (`GridSearchCV`), curva
de validação e escolha justificada do modelo final de escopo.

In [ ]:
# 🔨 PARA GERAR A CURVA DE VALIDAÇÃO
busca_escopo = GridSearchCV(
    Pipeline([("vetorizador", tfidf_caracteres()), ("classificador", LogisticRegression(max_iter=2000))]),
    param_grid={"classificador__C": [0.1, 1, 10, 100, 1000, 10000]},
    scoring="f1_macro", cv=cv, return_train_score=True,n_jobs=1,
)
busca_escopo.fit(treino["texto"], treino["escopo"])
clf_escopo = busca_escopo.best_estimator_

res = pd.DataFrame(busca_escopo.cv_results_)
ax = plt.gca()
ax.semilogx(res["param_classificador__C"].astype(float), res["mean_train_score"], "o-", color=CORES[0], label="treino")
ax.semilogx(res["param_classificador__C"].astype(float), res["mean_test_score"], "s-", color=CORES[3], label="validação")
ax.set_xlabel("C (= 1/λ)")
ax.set_ylabel("F1 macro")
ax.set_title("Escopo: curva de validação para C")
ax.legend()
plt.show()
print("Melhor C:", busca_escopo.best_params_["classificador__C"],
      f"| F1 macro (CV) = {busca_escopo.best_score_:.3f}")

In [ ]:
# TESTANDO EMBEDDINGS-MLP na fase de escolha
'''

busca_escopo = GridSearchCV(
    Pipeline([("vetorizador", EmbeddingsSentenca()), ("classificador", MLPClassifier(hidden_layer_sizes=(64,),
            max_iter=600,
            random_state=RANDOM_STATE
        ))]),
    param_grid={"classificador__alpha": [1e-4, 1e-3, 1e-2, 1e-1]},
    scoring="f1_macro", cv=cv, n_jobs=1,
)
busca_escopo.fit(treino["texto"], treino["escopo"])
clf_escopo = busca_escopo.best_estimator_

res = pd.DataFrame(busca_escopo.cv_results_)
ax = plt.gca()
ax.semilogx(res["param_classificador__alpha"].astype(float), res["mean_train_score"], "o-", color=CORES[0], label="treino")
ax.semilogx(res["param_classificador__alpha"].astype(float), res["mean_test_score"], "s-", color=CORES[3], label="validação")
ax.set_xlabel("α")
ax.set_ylabel("F1 macro")
ax.set_title("Escopo: curva de validação para α")
ax.legend()
plt.show()
print("Melhor α:", busca_escopo.best_params_["classificador__alpha"],
      f"| F1 macro (CV) = {busca_escopo.best_score_:.3f}")
      
'''

📝 RELATÓRIO: matriz de confusão (validação cruzada), taxa de falso aceite e de
falsa recusa, e análise dos erros. Qual erro é mais grave **no seu domínio**?

In [ ]:
prev_cv = cross_val_predict(clf_escopo, treino["texto"], treino["escopo"], cv=cv)
ConfusionMatrixDisplay.from_predictions(treino["escopo"], prev_cv, labels=CLASSES_ESCOPO,
                                        cmap="Blues", colorbar=False)
plt.grid(False)
plt.show()
treino.assign(previsto=prev_cv).query("escopo != previsto")[["texto", "escopo", "previsto", "origem"]]

---
## 5. Classificador de tema (intenções do domínio)

🔨 DONE: mesma metodologia, treinando **só** com as mensagens do domínio.

In [ ]:
treino_dom = treino[treino["escopo"] == "dominio"]
tab_tema = comparar(treino_dom["texto"], treino_dom["intencao"])
tab_tema.round(3)

In [ ]:
grafico_comparacao(tab_tema, "Classificador de tema")

In [ ]:
# 🔨 DONE: ajuste de hiperparâmetros e escolha do modelo de tema
busca_tema = GridSearchCV(
    Pipeline([("vetorizador", tfidf_caracteres()), ("classificador", LogisticRegression(max_iter=2000))]),
    param_grid={"classificador__C": [0.5, 1, 5, 10, 15]},
    scoring="f1_macro", cv=cv, return_train_score=True, n_jobs=1,
)
busca_tema.fit(treino_dom["texto"], treino_dom["intencao"])
clf_tema = busca_tema.best_estimator_

res = pd.DataFrame(busca_tema.cv_results_)
ax = plt.gca()
ax.semilogx(res["param_classificador__C"].astype(float), res["mean_train_score"], "o-", color=CORES[0], label="treino")
ax.semilogx(res["param_classificador__C"].astype(float), res["mean_test_score"], "s-", color=CORES[3], label="validação")
ax.set_xlabel("C (= 1/λ)")
ax.set_ylabel("F1 macro")
ax.set_title("Tema: curva de validação para C")
ax.legend()
plt.show()
print("Melhor C:", busca_tema.best_params_["classificador__C"],
      f"| F1 macro (CV) = {busca_tema.best_score_:.3f}")

In [ ]:
prev_tema = cross_val_predict(clf_tema, treino_dom["texto"], treino_dom["intencao"], cv=cv)
ConfusionMatrixDisplay.from_predictions(treino_dom["intencao"], prev_tema, labels=config.nomes_intencoes,
                                        cmap="Reds", colorbar=False, xticks_rotation=45)
plt.grid(False)
plt.show()
print(classification_report(treino_dom["intencao"], prev_tema, zero_division=0))
treino_dom.assign(previsto=prev_tema).query("intencao != previsto")[["texto", "intencao", "previsto", "origem"]]

---
## 6. Limiar de confiança

🔨 DONE: escolha `roteamento.limiar_tema` (no `grupo.yaml`) com base nas
probabilidades da validação cruzada, como na seção 9 do exemplo. Justifique.

In [ ]:
# 🔨 DONE
proba_cv = cross_val_predict(clf_tema, treino_dom["texto"], treino_dom["intencao"], cv=cv,
                             method="predict_proba")
classes = np.array(sorted(treino_dom["intencao"].unique()))
confianca = proba_cv.max(axis=1)
acertou = classes[proba_cv.argmax(axis=1)] == treino_dom["intencao"].to_numpy()

bins = np.linspace(0, 1, 21)
plt.hist(confianca[acertou], bins=bins, alpha=0.7, color=CORES[2], label=f"acertos ({acertou.sum()})")
plt.hist(confianca[~acertou], bins=bins, alpha=0.8, color=CORES[3], label=f"erros ({(~acertou).sum()})")
plt.axvline(config.limiar_tema, color="k", ls="--", lw=1, label=f"limiar = {config.limiar_tema}")
plt.xlabel("maior probabilidade do classificador de tema")
plt.ylabel("frases")
plt.title("Confiança das previsões (validação cruzada)")
plt.legend()
plt.show()

linhas = []
for limiar in [0.0, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    respondidas = confianca >= limiar
    linhas.append({"limiar": limiar,
                   "% respondidas direto": respondidas.mean(),
                   "acurácia nas respondidas": acertou[respondidas].mean() if respondidas.any() else np.nan,
                   "erros evitados": int((~acertou & ~respondidas).sum()),
                   "acertos que viraram pergunta": int((acertou & ~respondidas).sum())})
pd.DataFrame(linhas).round(3)

---
## 7. Salvar os modelos finais

In [ ]:
salvar_classificadores(clf_escopo, clf_tema, config)
config = carregar_config("grupo.yaml")  # relê, caso o limiar tenha sido alterado

---
## 8. O chatbot funcionando (com o LLM do OpenRouter)

🔨 DONE: reúna os documentos em `documentos/` (uma subpasta por intenção + `geral/`,
com o `FONTES.md`), ajuste `persona` e `contato` no `grupo.yaml` e mostre conversas
que ilustrem acertos **e** limitações do bot (inclusive perguntas cuja resposta
não está nos documentos). Se o grupo alterou o grafo (`chatbot/grafo.py`),
explique aqui o que mudou.

Antes: configure a chave no `.env` (ou nos *Secrets* do Colab) e rode
`python scripts/testar_llm.py`.

In [ ]:
from chatbot.conhecimento import BaseConhecimento
from chatbot.llm import ClienteLLM, ErroLLM, prompt_intencao

base = BaseConhecimento.carregar_ou_criar(config)   # cria o índice na 1ª vez
display(base.resumo())

# 🔨 DONE: teste a busca com perguntas do seu domínio (não gasta créditos)
pergunta, intencao = "O que tem no almoço hoje?", config.nomes_intencoes[0]
for tr, sim in base.buscar(pergunta, intencao=intencao):
    print(pergunta, intencao)
    print(f"{sim:.2f}  {tr.fonte} | {tr.texto[:80]!r}\n")

In [ ]:
# o prompt que o LLM recebe para essa pergunta
for m in prompt_intencao(config, intencao, pergunta, trechos=base.buscar(pergunta, intencao=intencao)):
    print(f"--- {m['role']} ---\n{m['content']}\n")

In [ ]:
try:
    llm = ClienteLLM.da_config(config)
except ErroLLM as erro:
    print("Sem LLM:", erro)
    llm = None

conversa = Conversa(config, clf_escopo, clf_tema, llm=llm, base=base)
for m in ["oi!", "Qual a janta hoje no RU?", "obrigado"]:
    e = conversa.enviar(m)
    print(f"você: {m}\n  [rota={e['rota']} | resposta {e['fonte_resposta']}]\n  bot: {e['resposta']}\n")
if llm:
    print("Uso do LLM:", llm.uso)

---
## 9. Avaliação final no teste (uma única vez)

📝 RELATÓRIO: tabela dos três níveis, matriz de confusão ponta a ponta e análise
dos erros (separando, se possível, por origem das frases).

In [ ]:
clf_escopo, clf_tema = carregar_classificadores(config)
grafo = construir_grafo(config, clf_escopo, clf_tema)
resultado, previsoes = avaliar(config, clf_escopo, clf_tema, grafo, teste)
tab, extras = tabela_resumo(resultado)
display(tab.round(3))
display(extras.round(3))
previsoes.loc[~previsoes["acertou"], ["texto", "real", "rota", "confianca_escopo", "confianca_intencao"]]

---
## 10. Uso de IA generativa

🔨 DONE: confira se **todo** uso de IA do grupo (geração de dados, LLM do bot,
código, escrita, ideias, depuração...) está no `USO_DE_IA.md` e será copiado para a
seção própria do artigo e mencionado no vídeo. **Uso não declarado = nota zero
(plágio).**

---
## 11. Verificação da entrega

Deve terminar com **0 erro(s)**.

In [ ]:
!{sys.executable} scripts/verificar_entrega.py